# 🧪 Research Experiment: FocusedLoss vs. MSE for Extreme Wave Prediction

This notebook implements a rigorous scientific comparison between a standard `MSELoss` baseline and our novel `FocusedLoss` function for predicting extreme wave events.

**Objective:** To demonstrate that `FocusedLoss` significantly improves the model's accuracy on rare, high-impact (extreme) wave events compared to a model trained with standard MSE.

**Methodology:**
1.  **Identical Setup:** We will use the exact same model architecture (`UConvLSTM`), data, and hyperparameters for both experiments.
2.  **Experiment 1 (Baseline):** Train the model using standard `nn.MSELoss()`.
3.  **Experiment 2 (Proposed):** Train an identical, re-initialized model using our custom `FocusedLoss()`.
4.  **Evaluation:** We will compare both models on a held-out test set using a `ComprehensiveEvaluator`. The key metric will be **`Extreme_RMSE`**, which measures the error *only* on wave events above our `EXTREME_THRESHOLD`.

## 1. Imports and Configuration

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

import os
import pickle
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
import shutil
import glob
from multiprocessing import Pool, cpu_count

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

# =============================================================================
# MASTER CONFIGURATION
# =============================================================================

# Core Parameters
LOOKBACK_HOURS = 96
FORECAST_HORIZON_HOURS = 24

# File Paths (Update these to your local paths)
RAW_DATA_PATH = r'/home/aidl/Wave-Prediction/dAtA/cmems_mod_ibi_wav_my_0.027deg_PT1H-i_multi-vars_11.00W-8.53W_38.50N-40.47N_2020-01-01-2023-12-30.nc'
STATIC_DATA_PATH = r'/home/aidl/Wave-Prediction/dAtA/cmems_GEBCO_resampled.nc'
BASE_DIR = r'./wave_experiment'
LOG_DIR = os.path.join(BASE_DIR, 'logs')
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, f'processed_data_lookback_{LOOKBACK_HOURS}_forecast_{FORECAST_HORIZON_HOURS}_static')
MODEL_SAVE_DIR = os.path.join(BASE_DIR, 'models') # Checkpoints will go here

# Feature Engineering
#
# --- YOUR REQUESTS ARE INCLUDED HERE ---
# 1. 'VHM0' is included in this list.
# 2. Variable names 'VSDX', 'VSDY' are correct.
#
VARS_TO_USE = ['VCMX', 'VHM0', 'VSDX', 'VSDY', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']
TARGET_VAR = 'VCMX'
# Static var names (fixed from our conversation)
STATIC_DEPTH_VAR = 'deptho' 
STATIC_ASPECT_VAR = 'aspect' # (We know this is missing, code will handle it)
# Total channels = 11 time-varying + 2 static
INPUT_CHANNELS = len(VARS_TO_USE) + 2

# Training Hyperparameters
LEARNING_RATE = 1e-5
BATCH_SIZE = 8
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5
DROPOUT_RATE = 0.2  # <-- NEW: Dropout rate you requested

# Research Parameters
EXTREME_THRESHOLD = 4.0  # VCMX threshold for extreme events
FOCUSED_WEIGHT = 50.0    # Penalty multiplier for extreme events

# System Configuration
# Use 2 fewer cores to keep PC responsive
NUM_PREPROCESSING_WORKERS = os.cpu_count() - 2 if os.cpu_count() > 2 else 1
NUM_DATALOADER_WORKERS = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"

# Create directories
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("--- Research Experiment Configuration (Pytorch Lightning Version) ---")
print(f"VHM0 Included: {'VHM0' in VARS_TO_USE}")
print(f"Dropout Rate: {DROPOUT_RATE}")
print(f"Input Channels: {INPUT_CHANNELS} ({len(VARS_TO_USE)} time-varying + 2 static)")
print(f"Using Device: {device} (Accelerator: {ACCELERATOR})")
print(f"Preprocessing Workers: {NUM_PREPROCESSING_WORKERS}")
print("---------------------------------------------------------------------")

/home/aidl/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Research Experiment Configuration (Pytorch Lightning Version) ---
VHM0 Included: True
Dropout Rate: 0.2
Input Channels: 13 (11 time-varying + 2 static)
Using Device: cuda (Accelerator: gpu)
Preprocessing Workers: 38
---------------------------------------------------------------------


## 2. Loss Functions

Here we define the two loss functions for our comparison: `FocusedLoss` and standard `MSELoss`.

In [2]:
class FocusedLoss(nn.Module):
    """
    A custom Weighted Mean Squared Error (WMSE) loss function that applies
    a heavy penalty to errors on high-value targets (extreme events).
    """
    
    def __init__(self, threshold=4.0, weight=50.0):
        super(FocusedLoss, self).__init__()
        self.threshold = threshold
        self.weight = weight
        # Use 'none' reduction to get element-wise errors
        self.mse = nn.MSELoss(reduction='none')
        print(f"FocusedLoss initialized: threshold={self.threshold}m, weight={self.weight}x")
    
    def forward(self, y_pred, y_true):
        # 1. Calculate the standard, element-wise squared error
        element_wise_error = self.mse(y_pred, y_true)
        
        # 2. Create the weight map based on the *true* values
        weights = torch.ones_like(y_true)
        weights = torch.where(y_true >= self.threshold, self.weight, weights)
        
        # 3. Apply the weights to the errors
        weighted_error = element_wise_error * weights
        
        # 4. Return the mean of the weighted errors
        return weighted_error.mean()

print("✅ FocusedLoss class defined.")

✅ FocusedLoss class defined.


## 3. Data Preprocessing & Loading

This section contains the code to process the raw `.nc` files into samples and the `Dataset` class to load them.

**Note:** The `run_preprocessing()` cell only needs to be run once. If you have already processed the data, you can skip it.

In [3]:
# --- This function must be at the top level for multiprocessing ---
def worker_process_sample(args):
    """
    This function processes and saves a single sample.
    It's designed to be run in a separate process.
    """
    t, config = args
    try:
        with xr.open_dataset(config['raw_data_path']) as ds_time:
            start_idx = t
            end_idx = t + config['lookback_hours']
            input_slice = ds_time.isel(time=slice(start_idx, end_idx))
            
            target_idx = t + config['window_size'] - 1
            target_slice = ds_time[config['target_var']].isel(time=target_idx)

            input_data_list = []
            for var in config['vars_to_use']:
                data = input_slice[var].values 
                original_shape = data.shape
                data_scaled = config['scaler'].transform(data.flatten().reshape(-1, 1)).reshape(original_shape)
                input_data_list.append(data_scaled)
            
            for static_data in config['static_data_tiled_list']:
                input_data_list.append(static_data)

            input_tensor = np.stack(input_data_list, axis=-1)
            input_tensor = np.transpose(input_tensor, (3, 0, 1, 2))
            
            target_data = target_slice.values
            target_scaled = config['target_scaler'].transform(target_data.reshape(-1, 1)).reshape(target_data.shape)
            target_tensor = target_scaled.astype(np.float32)
            
            sample_path = os.path.join(config['processed_data_dir'], f"sample_{t:05d}.pkl")
            with open(sample_path, 'wb') as f:
                pickle.dump({'input': input_tensor.astype(np.float32), 'target': target_tensor}, f)
        return (t, "Success")
    except Exception as e:
        return (t, f"Error: {e}")

class DataPreprocessor:
    def __init__(self, raw_data_path, static_data_path, processed_data_dir, lookback_hours, forecast_horizon_hours, target_var, vars_to_use, static_depth_var, static_aspect_var):
        self.raw_data_path = raw_data_path
        self.static_data_path = static_data_path
        self.processed_data_dir = processed_data_dir
        self.lookback_hours = lookback_hours
        self.forecast_horizon_hours = forecast_horizon_hours
        self.target_var = target_var
        self.vars_to_use = vars_to_use
        self.static_depth_var = static_depth_var
        self.static_aspect_var = static_aspect_var
        
        self.scaler = MinMaxScaler()
        self.target_scaler = MinMaxScaler()
        self.static_scalers = {}
        os.makedirs(self.processed_data_dir, exist_ok=True)

    def fit_scalers_chunked(self, time_chunk_size=5000):
        print(f"Starting chunked scaler fitting (FASTER chunk size: {time_chunk_size})...")
        try:
            ds_time = xr.open_dataset(self.raw_data_path)
            total_times = len(ds_time.time)
            for var in self.vars_to_use:
                if var not in ds_time.data_vars:
                    raise KeyError(f"Variable '{var}' from VARS_TO_USE not found in the file!")
            
            for t_start in tqdm(range(0, total_times, time_chunk_size), desc="Fitting Scalers"):
                t_end = min(t_start + time_chunk_size, total_times)
                ds_chunk = ds_time.isel(time=slice(t_start, t_end)).load()
                
                data_list = [ds_chunk[var].values.flatten().reshape(-1, 1) for var in self.vars_to_use]
                all_data_chunk = np.concatenate(data_list, axis=0)
                self.scaler.partial_fit(all_data_chunk)
                
                target_data_chunk = ds_chunk[self.target_var].values.flatten().reshape(-1, 1)
                self.target_scaler.partial_fit(target_data_chunk)
            ds_time.close()
        except Exception as e:
            print(f"Error during chunked time-scaler fitting: {e}")
            return False

        try:
            ds_static = xr.open_dataset(self.static_data_path)
            if self.static_depth_var in ds_static:
                ds_static = ds_static.rename({self.static_depth_var: 'depth_processed'})
                ds_static['depth_processed'] = xr.where(ds_static['depth_processed'] > 0, -ds_static['depth_processed'], ds_static['depth_processed'])
            else:
                raise KeyError(f"Static variable '{self.static_depth_var}' not found in static file.")
            
            if self.static_aspect_var in ds_static:
                ds_static['cos_aspect'] = np.cos(np.radians(ds_static[self.static_aspect_var]))
            else:
                print(f"Info: '{self.static_aspect_var}' not found. Filling 'cos_aspect' with zeros.")
                ds_static['cos_aspect'] = xr.full_like(ds_static['depth_processed'], 0.0)
            
            for var in ['depth_processed', 'cos_aspect']:
                scaler = MinMaxScaler()
                data = ds_static[var].values.reshape(-1, 1)
                scaler.fit(data)
                self.static_scalers[var] = scaler
            ds_static.close()
        except Exception as e:
            print(f"Error during static-scaler fitting: {e}")
            return False

        print("Scaler fitting complete. Saving to disk...")
        with open(os.path.join(self.processed_data_dir, 'scaler_all_vars.pkl'), 'wb') as f: pickle.dump(self.scaler, f)
        with open(os.path.join(self.processed_data_dir, f'scaler_{self.target_var}.pkl'), 'wb') as f: pickle.dump(self.target_scaler, f)
        with open(os.path.join(self.processed_data_dir, 'scalers_static.pkl'), 'wb') as f: pickle.dump(self.static_scalers, f)
        return True

    def preprocess_and_save(self, num_workers=None):
        if not self.fit_scalers_chunked():
            print("Halting preprocessing due to scaler fitting error.")
            return

        print("Loading fitted scalers from disk...")
        with open(os.path.join(self.processed_data_dir, 'scaler_all_vars.pkl'), 'rb') as f: scaler = pickle.load(f)
        with open(os.path.join(self.processed_data_dir, f'scaler_{self.target_var}.pkl'), 'rb') as f: target_scaler = pickle.load(f)
        with open(os.path.join(self.processed_data_dir, 'scalers_static.pkl'), 'rb') as f: static_scalers = pickle.load(f)
            
        print("Loading and scaling static data for all workers...")
        ds_static = xr.open_dataset(self.static_data_path)
        if self.static_depth_var in ds_static:
            ds_static = ds_static.rename({self.static_depth_var: 'depth_processed'})
            ds_static['depth_processed'] = xr.where(ds_static['depth_processed'] > 0, -ds_static['depth_processed'], ds_static['depth_processed'])
        else:
             raise KeyError(f"Static variable '{self.static_depth_var}' not found in static file.")
        if self.static_aspect_var in ds_static:
            ds_static['cos_aspect'] = np.cos(np.radians(ds_static[self.static_aspect_var]))
        else:
            ds_static['cos_aspect'] = xr.full_like(ds_static['depth_processed'], 0.0)

        static_data_tiled_list = []
        for var_name, scaler_obj in static_scalers.items():
            data = ds_static[var_name].values
            data_scaled = scaler_obj.transform(data.reshape(-1, 1)).reshape(data.shape)
            data_tiled = np.tile(np.expand_dims(data_scaled, axis=0), (self.lookback_hours, 1, 1)).astype(np.float32)
            static_data_tiled_list.append(data_tiled)
        ds_static.close()
        print("Static data processed and ready.")

        if num_workers is None:
            num_workers = cpu_count()
        print(f"Setting num_workers to: {num_workers}")
        
        with xr.open_dataset(self.raw_data_path) as ds_time:
            total_hours = len(ds_time.time)
        
        window_size = self.lookback_hours + self.forecast_horizon_hours
        tasks_to_process = list(range(total_hours - window_size + 1))
        total_samples = len(tasks_to_process)
        
        config = {
            'raw_data_path': self.raw_data_path, 'processed_data_dir': self.processed_data_dir,
            'lookback_hours': self.lookback_hours, 'window_size': window_size,
            'target_var': self.target_var, 'vars_to_use': self.vars_to_use,
            'scaler': scaler, 'target_scaler': target_scaler,
            'static_data_tiled_list': static_data_tiled_list
        }

        print(f"Starting parallel processing of {total_samples} samples with {num_workers} workers...")
        worker_args = [(t, config) for t in tasks_to_process]
        
        num_saved = 0
        with Pool(processes=num_workers) as pool:
            with tqdm(total=total_samples, desc="Saving Samples (Parallel)") as pbar:
                for t, result in pool.imap_unordered(worker_process_sample, worker_args):
                    if result == "Success": num_saved += 1
                    else: print(f"Warning: Failed to process sample {t}. Error: {result}")
                    pbar.update(1)

        print(f"Successfully generated and saved {num_saved} samples.")

print("✅ DataPreprocessor (Parallel) defined.")

✅ DataPreprocessor (Parallel) defined.


### ⬇️ **Run This Cell to Preprocess Data** ⬇️

**Run this cell ONCE.** It may take a long time. After it's done, the `processed_data` directory will be populated, and you won't need to run it again.

In [4]:
# # --- Run This Cell Once to Preprocess Data ---
# #
# # Check if preprocessing is already done
# if not os.path.exists(os.path.join(PROCESSED_DATA_DIR, 'scalers_static.pkl')):
#     print("Processed data not found. Starting preprocessing...")
    
#     preprocessor = DataPreprocessor(
#         raw_data_path=RAW_DATA_PATH,
#         static_data_path=STATIC_DATA_PATH,
#         processed_data_dir=PROCESSED_DATA_DIR,
#         lookback_hours=LOOKBACK_HOURS,
#         forecast_horizon_hours=FORECAST_HORIZON_HOURS,
#         target_var=TARGET_VAR,
#         vars_to_use=VARS_TO_USE,
#         static_depth_var=STATIC_DEPTH_VAR,
#         static_aspect_var=STATIC_ASPECT_VAR
#     )
    
#     # Run the new parallel function
#     preprocessor.preprocess_and_save(num_workers=NUM_PREPROCESSING_WORKERS)
    
#     print("--- DATA PREPROCESSING FINISHED ---")
# else:
#     print("--- Data already preprocessed. Skipping. ---")

In [5]:
class WaveForecastDataset(Dataset):
    """
    PyTorch Dataset for loading the preprocessed wave forecast samples.
    """
    def __init__(self, processed_data_dir, split='train', train_split=0.7, val_split=0.15):
        self.processed_data_dir = processed_data_dir
        self.file_paths = sorted(glob.glob(os.path.join(processed_data_dir, "sample_*.pkl")))
        
        if not self.file_paths:
            raise FileNotFoundError(f"No '.pkl' samples found in {processed_data_dir}. "
                                    f"Did you run the preprocessing cell?\n")

        # Determine file splits
        total_samples = len(self.file_paths)
        train_end = int(total_samples * train_split)
        val_end = train_end + int(total_samples * val_split)

        if split == 'train':
            self.file_paths = self.file_paths[:train_end]
        elif split == 'val':
            self.file_paths = self.file_paths[train_end:val_end]
        elif split == 'test':
            self.file_paths = self.file_paths[val_end:]
        else:
            raise ValueError("split must be 'train', 'val', or 'test'")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        with open(self.file_paths[idx], 'rb') as f:
            data = pickle.load(f)
        # Input: (Channels, Time, Lat, Lon), Target: (Lat, Lon)
        return torch.from_numpy(data['input']), torch.from_numpy(data['target'])

class WaveForecastDataModule(pl.LightningDataModule):
    """
    PyTorch Lightning DataModule to handle all data loading.
    """
    def __init__(self, processed_data_dir, batch_size=32, num_workers=4):
        super().__init__()
        self.processed_data_dir = processed_data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.save_hyperparameters()

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            self.train_dataset = WaveForecastDataset(self.processed_data_dir, split='train')
            self.val_dataset = WaveForecastDataset(self.processed_data_dir, split='val')
        if stage == 'test' or stage is None:
            self.test_dataset = WaveForecastDataset(self.processed_data_dir, split='test')

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, 
                          shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, 
                        shuffle=False, num_workers=self.num_workers, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, 
                        shuffle=False, num_workers=self.num_workers, pin_memory=True)

print("✅ WaveForecastDataset and WaveForecastDataModule defined.")

✅ WaveForecastDataset and WaveForecastDataModule defined.


## 4. Model Architecture (U-ConvLSTM)

This is the `UConvLSTM` model. It will be used for *both* experiments.

##Lightning Module (From Your Notebook

In [6]:
#
# --- NEW, FINAL-FIX CELL 6 (Model Architecture) ---
# This version adds a final resize step to fix the 88 vs 90 error.
#

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.bias = bias
        self.conv = nn.Conv2d(in_channels=self.input_dim + self.hidden_dim,
                              out_channels=4 * self.hidden_dim,
                              kernel_size=self.kernel_size,
                              padding=self.padding, bias=self.bias)
    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next
    def init_hidden(self, batch_size, image_size):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device))

class Encoder(nn.Module):
    def __init__(self, in_channels, hidden_dims, kernel_size, dropout_rate, bias=True):
        super(Encoder, self).__init__()
        self.layers = nn.ModuleList()
        self.hidden_dims = hidden_dims
        for i in range(len(hidden_dims)):
            in_dim = in_channels if i == 0 else hidden_dims[i-1]
            self.layers.append(ConvLSTMCell(in_dim, hidden_dims[i], (kernel_size, kernel_size), bias))
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout2d(p=dropout_rate)

    def forward(self, x, seq_len):
        b, _, t, h, w = x.shape
        hidden_states = []
        skip_connections = []
        for layer_idx in range(len(self.hidden_dims)):
            hidden_states.append(self.layers[layer_idx].init_hidden(b, (h, w)))
        
        x_in = x
        for layer_idx, layer in enumerate(self.layers):
            h, c = hidden_states[layer_idx]
            h_skip_init, c_skip_init = layer.init_hidden(b, (h.shape[2], h.shape[3]))
            h_skip, c_skip = h_skip_init, c_skip_init # Init for this layer
            
            for t_step in range(seq_len):
                h, c = layer(x_in[:, :, t_step, :, :], (h, c))
                if t_step == seq_len - 1: # Save last state
                    h_skip, c_skip = h, c
            
            h_for_skip = h_skip # Use the pre-dropout 'h' for the skip connection
            
            h = self.dropout(h) # Apply dropout to the main path
            
            skip_connections.append((h_for_skip, c_skip)) 
            hidden_states[layer_idx] = (h, c)
            
            if layer_idx < len(self.hidden_dims) - 1:
                x_in = self.pool(h) 
                x_in = x_in.unsqueeze(2).repeat(1, 1, t, 1, 1)
                
                new_h, new_w = x_in.shape[3], x_in.shape[4]
                for i in range(len(hidden_states)):
                    h_i, c_i = hidden_states[i]
                    h_i_new = F.interpolate(h_i, size=(new_h, new_w), mode='bilinear', align_corners=False)
                    c_i_new = F.interpolate(c_i, size=(new_h, new_w), mode='bilinear', align_corners=False)
                    hidden_states[i] = (h_i_new, c_i_new)
        return hidden_states, skip_connections

class Decoder(nn.Module):
    def __init__(self, in_channels, hidden_dims, encoder_hidden_dims, kernel_size, dropout_rate, bias=True):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList()
        self.hidden_dims = hidden_dims
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dropout = nn.Dropout2d(p=dropout_rate)
        
        encoder_hidden_dims = encoder_hidden_dims.copy()
        encoder_hidden_dims.reverse() # [16, 32, 64] -> [64, 32, 16]
        
        for i in range(len(hidden_dims)):
            if i == 0:
                in_dim = in_channels
            else:
                in_dim = hidden_dims[i-1] + encoder_hidden_dims[i]
            self.layers.append(ConvLSTMCell(in_dim, hidden_dims[i], (kernel_size, kernel_size), bias))

    def forward(self, x, seq_len, skip_connections):
        b, _, t, h, w = x.shape
        hidden_states = []
        for layer_idx in range(len(self.hidden_dims)):
            hidden_states.append(self.layers[layer_idx].init_hidden(b, (h, w)))
        
        x_in_current = x 
        h_prev = None 

        for layer_idx, layer in enumerate(self.layers):
            h, c = hidden_states[layer_idx] 
            
            if layer_idx > 0:
                x_in_upsample = self.upsample(h_prev) 
                skip_h, _ = skip_connections[-(layer_idx+1)]

                target_h, target_w = x_in_upsample.shape[2], x_in_upsample.shape[3]
                if skip_h.shape[2] != target_h or skip_h.shape[3] != target_w:
                    skip_h = F.interpolate(skip_h, size=(target_h, target_w), mode='bilinear', align_corners=False)

                x_in_cat = torch.cat([x_in_upsample, skip_h], dim=1)
                x_in_current = x_in_cat.unsqueeze(2).repeat(1, 1, t, 1, 1)

                new_h, new_w = x_in_current.shape[3], x_in_current.shape[4]
                for i in range(len(hidden_states)):
                    h_i, c_i = hidden_states[i]
                    h_i_new = F.interpolate(h_i, size=(new_h, new_w), mode='bilinear', align_corners=False)
                    c_i_new = F.interpolate(c_i, size=(new_h, new_w), mode='bilinear', align_corners=False)
                    hidden_states[i] = (h_i_new, c_i_new)
                
                h, c = hidden_states[layer_idx]

            for t_step in range(seq_len):
                h, c = layer(x_in_current[:, :, t_step, :, :], (h, c))
            
            h = self.dropout(h)
            hidden_states[layer_idx] = (h, c)
            h_prev = h 

        return h # Return the final 'h' from the last layer

class UConvLSTM_with_Dropout(nn.Module):
    def __init__(self, in_channels, out_channels=1,
                 encoder_hidden=[16, 32, 64],
                 decoder_hidden=[64, 32, 16],
                 kernel_size=3, dropout_rate=0.2):
        super(UConvLSTM_with_Dropout, self).__init__()
        
        self.encoder_hidden = encoder_hidden
        self.decoder_hidden = decoder_hidden
        
        self.encoder = Encoder(in_channels, self.encoder_hidden, kernel_size, dropout_rate)
        
        decoder_in_channels = self.encoder_hidden[-1]
        self.decoder = Decoder(decoder_in_channels, self.decoder_hidden, self.encoder_hidden, kernel_size, dropout_rate)
        
        self.final_conv = nn.Conv2d(in_channels=self.decoder_hidden[-1],
                                    out_channels=out_channels,
                                    kernel_size=1, padding=0, bias=True)
        self.output_activation = nn.Sigmoid()
        
    def forward(self, x):
        # Get target shape from input x
        # x shape is (B, C_in, T, H, W)
        target_h, target_w = x.shape[3], x.shape[4]
        
        b, c, t, h, w = x.shape
        _, skip_connections = self.encoder(x, seq_len=t)
        
        last_encoder_h, _ = skip_connections[-1]
        
        decoder_input = last_encoder_h.unsqueeze(2).repeat(1, 1, t, 1, 1)
        decoder_output_h = self.decoder(decoder_input, seq_len=t, skip_connections=skip_connections)
        
        # decoder_output_h is [B, 16, 36, 44] (or similar)
        prediction = self.final_conv(decoder_output_h)
        # prediction is [B, 1, 36, 44]

        #
        # --- THIS IS THE FIX ---
        #
        # If the prediction size doesn't match the target size, resize it.
        # This fixes the 88 vs 90 (or 36 vs 73, etc.) error.
        if prediction.shape[2] != target_h or prediction.shape[3] != target_w:
            prediction = F.interpolate(
                prediction, 
                size=(target_h, target_w), 
                mode='bilinear', 
                align_corners=False
            )
        # prediction is now [B, 1, 73, 90] (matches target)
        #
        # --- END FIX ---
        #
        
        prediction = self.output_activation(prediction)
        return prediction.squeeze(1) # Final shape [B, 73, 90]

print("✅ UConvLSTM_with_Dropout (FINAL FIX) and helper classes defined.")

✅ UConvLSTM_with_Dropout (FINAL FIX) and helper classes defined.


In [7]:
class WaveNet(pl.LightningModule):
    def __init__(self, model, criterion, learning_rate):
        super().__init__()
        self.model = model
        self.criterion = criterion
        self.learning_rate = learning_rate
        self.save_hyperparameters(ignore=['model', 'criterion'])

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = self.criterion(y_hat, y)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = self.criterion(y_hat, y)
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = self.criterion(y_hat, y)
        self.log('test_loss', loss, logger=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3),
                'monitor': 'val_loss',
            }
        }

print("✅ WaveNet (LightningModule) defined.")

✅ WaveNet (LightningModule) defined.


In [8]:
# Create the DataModule
data_module = WaveForecastDataModule(
    processed_data_dir=PROCESSED_DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_DATALOADER_WORKERS
)

# This will call data_module.setup()
data_module.prepare_data()

print("✅ DataModule is ready.")

✅ DataModule is ready.


In [9]:
print(f"\n--- 🧪 STARTING EXPERIMENT 1: PROPOSED (FocusedLoss) ---")

# 1. Init a NEW Model
model_focused = UConvLSTM_with_Dropout(
    in_channels=INPUT_CHANNELS,
    dropout_rate=DROPOUT_RATE
)

# 2. Init Loss Function
loss_fn_focused = FocusedLoss(
    threshold=EXTREME_THRESHOLD,
    weight=FOCUSED_WEIGHT
)

# 3. Init Lightning Module
lit_model_focused = WaveNet(
    model=model_focused,
    criterion=loss_fn_focused,
    learning_rate=LEARNING_RATE
)

# 4. Init Callbacks
checkpoint_callback_focused = ModelCheckpoint(
    dirpath=MODEL_SAVE_DIR,
    filename='proposed-focused-{epoch:02d}-{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    mode='min'
)

logger_focused = TensorBoardLogger(LOG_DIR, name='proposed_FocusedLoss')

# 5. Init Trainer
trainer_focused = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator=ACCELERATOR,
    devices=2, # Using 2 GPUs
    strategy="ddp_notebook", # Notebook-compatible DDP
    precision="16-mixed",
    accumulate_grad_batches=4,
    callbacks=[checkpoint_callback_focused, early_stop_callback],
    logger=logger_focused,
    log_every_n_steps=10
)

# 6. Run Training
trainer_focused.fit(lit_model_focused, datamodule=data_module)

print(f"--- ✅ EXPERIMENT 1 (FocusedLoss) FINISHED ---")
print(f"Best FocusedLoss model checkpoint: {checkpoint_callback_focused.best_model_path}")

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



--- 🧪 STARTING EXPERIMENT 1: PROPOSED (FocusedLoss) ---
FocusedLoss initialized: threshold=4.0m, weight=50.0x


Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 2 processes
----------------------------------------------------------------------------------------------------

2025-11-06 14:19:19.461928: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762418959.492212 1688189 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762418959.500538 1688189 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-06 14:19:19.529639: I tensorf

Epoch 0:  46%|████████████████████████▋                             | 697/1528 [1:03:16<1:15:25,  0.18it/s, v_num=8, train_loss_step=nan.0]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [ ]:
#
# --- This cell defines the evaluator and runs the final test ---
#

class ComprehensiveEvaluator:
    """
    This evaluator loads the models and scalers to calculate
    the final metrics, including the critical 'Extreme_RMSE'.
    """
    def __init__(self, processed_data_dir, target_var, extreme_threshold, device):
        self.device = device
        self.extreme_threshold = extreme_threshold
        self.results = {}
        
        # Load the target scaler to inverse-transform predictions back to meters
        scaler_path = os.path.join(processed_data_dir, f'scaler_{target_var}.pkl')
        with open(scaler_path, 'rb') as f:
            self.target_scaler = pickle.load(f)
            
    def evaluate_model(self, lit_model, test_loader, model_name):
        print(f"Evaluating {model_name} on test set...")
        lit_model.to(self.device)
        lit_model.eval()

        all_preds_scaled = []
        all_trues_scaled = []

        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Testing {model_name}"):
                x, y = batch
                x = x.to(self.device)
                y_hat_scaled = lit_model(x)
                
                all_preds_scaled.append(y_hat_scaled.cpu().numpy())
                all_trues_scaled.append(y.cpu().numpy())

        # Concatenate all batches
        y_pred_scaled = np.concatenate(all_preds_scaled, axis=0)
        y_true_scaled = np.concatenate(all_trues_scaled, axis=0)
        
        # Reshape for scaler: (num_samples * lat * lon, 1)
        y_pred_flat_scaled = y_pred_scaled.reshape(-1, 1)
        y_true_flat_scaled = y_true_scaled.reshape(-1, 1)

        # Inverse transform to get real-world values (meters)
        y_pred = self.target_scaler.inverse_transform(y_pred_flat_scaled)
        y_true = self.target_scaler.inverse_transform(y_true_flat_scaled)
        
        # --- Calculate all metrics ---
        metrics = {}
        metrics['Overall_RMSE (m)'] = np.sqrt(mean_squared_error(y_true, y_pred))
        metrics['Overall_MAE (m)'] = mean_absolute_error(y_true, y_pred)
        
        # --- Key Research Metric: Extreme Event Performance ---
        extreme_indices = np.where(y_true >= self.extreme_threshold)[0]
        
        if len(extreme_indices) > 0:
            extreme_y_true = y_true[extreme_indices]
            extreme_y_pred = y_pred[extreme_indices]
            metrics['**Extreme_RMSE (m)**'] = np.sqrt(mean_squared_error(extreme_y_true, extreme_y_pred))
            metrics['Extreme_MAE (m)'] = mean_absolute_error(extreme_y_true, extreme_y_pred)
        else:
            metrics['**Extreme_RMSE (m)**'] = np.nan
            metrics['Extreme_MAE (m)'] = np.nan
        
        self.results[model_name] = metrics
        return metrics

    def generate_comparison_table(self):
        print("\n" + "="*80)
        print("📊 FINAL MODEL COMPARISON (TEST SET RESULTS)")
        print("="*80)
        df = pd.DataFrame.from_dict(self.results, orient='index')
        print(df.to_string(float_format="%.4f"))
        print("="*80)
        print("Lower is better for all metrics.")
        return df

# --- RUN THE FINAL EVALUATION ---

print("Starting final evaluation on the TEST SET...")

# 1. Get the test dataloader
test_loader = data_module.test_dataloader()
print(f"Loaded {len(test_loader.dataset)} test samples.")

# 2. Init the Evaluator
evaluator = ComprehensiveEvaluator(
    processed_data_dir=PROCESSED_DATA_DIR,
    target_var=TARGET_VAR,
    extreme_threshold=EXTREME_THRESHOLD,
    device=device
)

# 3. Load best models and evaluate
try:
    # Load Best MSE Model
    best_mse_path = checkpoint_callback_mse.best_model_path
    print(f"Loading best MSE model from: {best_mse_path}")
    lit_model_mse_best = WaveNet.load_from_checkpoint(best_mse_path)
    evaluator.evaluate_model(lit_model_mse_best, test_loader, "Baseline_MSE")

    # Load Best FocusedLoss Model
    best_focused_path = checkpoint_callback_focused.best_model_path
    print(f"Loading best FocusedLoss model from: {best_focused_path}")
    lit_model_focused_best = WaveNet.load_from_checkpoint(best_focused_path)
    evaluator.evaluate_model(lit_model_focused_best, test_loader, "Proposed_FocusedLoss")

    # 4. Print the final results table
    results_df = evaluator.generate_comparison_table()

except Exception as e:
    print(f"\n--- ERROR DURING FINAL EVALUATION ---")
    print(f"Could not load or test models. Did the training runs complete?")
    print(f"Error details: {e}")

print("\n--- ✅ All Experiments Finished ---")

In [39]:
print(f"\n--- 🧪 STARTING EXPERIMENT 1: BASELINE (MSELoss) ---")

# 1. Init Model
model_mse = UConvLSTM_with_Dropout(
    in_channels=INPUT_CHANNELS,
    dropout_rate=DROPOUT_RATE
)

# 2. Init Loss Function
loss_fn_mse = nn.MSELoss()

# 3. Init Lightning Module
lit_model_mse = WaveNet(
    model=model_mse,
    criterion=loss_fn_mse,
    learning_rate=LEARNING_RATE
)

# 4. Init Callbacks
checkpoint_callback_mse = ModelCheckpoint(
    dirpath=MODEL_SAVE_DIR,
    filename='baseline-mse-{epoch:02d}-{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=True
)
early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    mode='min'
)
logger_mse = TensorBoardLogger(LOG_DIR, name='baseline_MSE')

# 5. Init Trainer
trainer_mse = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator=ACCELERATOR,
    devices=2,
    strategy="ddp_notebook", # <--- FIX
    callbacks=[checkpoint_callback_mse, early_stop_callback],
    logger=logger_mse,
    log_every_n_steps=10
)
# 6. Run Training
trainer_mse.fit(lit_model_mse, datamodule=data_module)

print(f"--- ✅ EXPERIMENT 1 (MSELoss) FINISHED ---")
print(f"Best MSE model checkpoint: {checkpoint_callback_mse.best_model_path}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



--- 🧪 STARTING EXPERIMENT 1: BASELINE (MSELoss) ---


RuntimeError: Lightning can't create new processes if CUDA is already initialized. Did you manually call `torch.cuda.*` functions, have moved the model to the device, or allocated memory on the GPU any other way? Please remove any such calls, or change the selected strategy. You will have to restart the Python kernel.